# Wheel-Based Pre-Oracle Filtering Tradeoff (Colab)

This notebook compares `mod30`, `mod210`, and `mod2310` as optional classical pre-oracle filters before Quantum Oracle Sketching (QOS).

Pipeline:
```text
raw classical candidates
    → wheel filter
    → filtered candidates
    → QOS sampling / oracle construction
```

This notebook measures candidate fractions and a simple oracle-call proxy. It does **not** claim quantum speedup; it isolates the classical filtering layer.

In [ ]:
# Clone your fork and move into repo root
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30

In [ ]:
# Imports (no path hacks needed since we're in repo root)
from modwheel import STANDARD_WHEELS
from pre_oracle_filter import oracle_call_proxy, pre_oracle_candidates_by_name

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Wheel density table

In [ ]:
rows = []
for name, wheel in STANDARD_WHEELS.items():
    rows.append({
        "wheel": name,
        "primes": "·".join(map(str, wheel.primes)),
        "modulus": wheel.modulus,
        "residue_count": wheel.residue_count,
        "density": wheel.density,
        "reduction": wheel.reduction,
    })

df = pd.DataFrame(rows)
df

## 2. Oracle-call proxy experiment

In [ ]:
N = 100_000
values = range(1, N + 1)
baseline = oracle_call_proxy(values)

bench_rows = []
for name, wheel in STANDARD_WHEELS.items():
    remaining = oracle_call_proxy(values, wheel)
    bench_rows.append({
        "wheel": name,
        "baseline": baseline,
        "remaining": remaining,
        "remaining_fraction": remaining / baseline,
        "reduction_fraction": 1 - remaining / baseline,
    })

bench = pd.DataFrame(bench_rows)
bench

## 3. Main figure (SVG + PNG)

In [ ]:
fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)

names = bench["wheel"].tolist()
fractions = bench["remaining_fraction"].to_numpy()

plt.figure(figsize=(7.5, 4.8))
plt.plot(names, fractions, marker='o')
plt.title("Wheel-Based Pre-Oracle Filtering: Diminishing Returns")
plt.xlabel("Wheel filter")
plt.ylabel("Remaining candidate fraction")
plt.ylim(0, 0.32)
plt.grid(True)

labels = [
    "8/30 = 26.7%\n73.3% excluded",
    "48/210 = 22.9%\n77.1% excluded",
    "480/2310 = 20.8%\n79.2% excluded",
]

for x, y, label in zip(names, fractions, labels):
    plt.annotate(label, (x, y), textcoords="offset points", xytext=(0, 10), ha="center")

svg_path = fig_dir / "modwheel_pre_oracle_tradeoff.svg"
png_path = fig_dir / "modwheel_pre_oracle_tradeoff.png"

plt.savefig(svg_path)
plt.savefig(png_path, dpi=200)
plt.show()

svg_path, png_path

## 4. Download figures

In [ ]:
from google.colab import files
files.download("figures/modwheel_pre_oracle_tradeoff.svg")
files.download("figures/modwheel_pre_oracle_tradeoff.png")

## 5. Interpretation

In [ ]:
print("""
mod30   → 26.7% remain (73.3% removed)
mod210  → 22.9% remain (77.1% removed)
mod2310 → 20.8% remain (79.2% removed)

Most reduction occurs at mod30.
Deeper wheels give diminishing returns.

Interpretation:
Classical filtering reduces oracle input size before QOS.
This introduces a tradeoff between preprocessing and oracle cost.
""")